## Basic Decoder-Only Transformer Implementation

In [1]:
# Installing files
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-21 16:28:16--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-03-21 16:28:16 (19.5 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [72]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { c:i for i, c in enumerate(chars)}
itos = { i:c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[id] for id in ids])

In [6]:
decode(encode('hello world'))

'hello world'

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)

In [8]:
split = int(0.9 * len(data))
train = data[:split]
valid = data[split:]

In [9]:
torch.manual_seed(100)
block_size = 8
batch_size = 4

def get_batch(split):
  assert split in ['train', 'valid'], "split is either train or valid"

  if split == "train":
    data = train
  else:
    data = valid

  idxs = data[torch.randint(0, len(data)-block_size, (batch_size, ))]

  x = torch.stack([data[idx:idx+block_size] for idx in idxs])
  y = torch.stack([data[idx+1:idx+block_size+1] for idx in idxs])

  return x, y

In [10]:
xb, yb = get_batch("train")
for i, j in zip(xb[0].tolist(), yb[0].tolist()):
  print(f"{decode([i])} -> {decode([j])}")

h -> e
e -> r
r -> ,
, ->  
  -> h
h -> e
e -> a
a -> r


In [111]:
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, vocab_size) # (vocab_size, vocab_size)

  def forward(self, x, targets = None):
    # x -> (B, T), targets -> (B, T)
    B, T = x.shape
    logits = self.embedding(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      logits = logits.view(B*T, vocab_size) # (B*T, vocab_size)
      targets = targets.view(B*T) # (B*T, )
      loss = F.cross_entropy(logits, targets) # scalar

    return logits, loss

  def generate(self, x, max_new_tokens=10):
    # x -> (B, T)
    for _ in range(max_new_tokens):
      logits, _ = self(x) # (B, T, vocab_size)
      """
        for each sample in the batch, look at the last character's logits and choose the next character.
        append the character at the end of each sample
      """
      logits = logits[:, -1, :] # (B, vocab_size)
      probs = F.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      # next_token = torch.argmax(logits, -1, keepdim=True) # (B, 1)
      x = torch.cat((x, next_token), dim=-1) # (B, T+1)

    return x

m = BigramLanguageModel()

In [136]:
# training
batch_size = 32
optimizer = optim.NAdam(m.parameters(), lr=1e-3)

for i in range(100):
  x, y = get_batch("train")

  # reset gradients
  optimizer.zero_grad(set_to_none=True)

  # forward pass
  logits, loss = m(x, y)

  # backward pass
  loss.backward()

  # update
  optimizer.step()

  # log statistics
  # print(f"Step {i+1} | loss = {loss}")

print(loss.item())

1.771436333656311


In [137]:
# generate output
xt = torch.tensor([encode("a")])
yt = m.generate(xt, 1000)

outputs = [decode(output.tolist()) for output in yt]
for output in outputs:
  print(output)

at vUhn:Fw?W!;gnj&TjWHO,aHFARIRIR:VTl3DyMO'IPrXoivAl:Vxjze;NMNFfoche QDl:
JUEuBir.
SdT?FKs?CheO$stfnyl:jyUo:Z
:v?&V$y? wt CVIwPwuLLRsRyGopeZ
ar!FCkALzJweCZ'Boc
HFur C!hea?Qx3$yLKhqSUr eanthel:VhqTt qY:

BL!peMst spear,'pe s,TBXYjl:OoperoB
wl
EzeCRIvhTper,st CVTtirzoOqE'fWqKxOFfbyFH'pa-HpJy,XBO!;a&r?QOAY,xbUWquZ
tiXngq?eEuy?Bd dDnyCb3&LtiqFVvx3PkGPh'fn3vAllpohny:
l:
'xc;jxi?MspeamefoFfKhMr,CirK?pen:kMkdher Waro;MUjTSqFUDY,.sPDY.MWHJweanvUDKpHUKIXWr,Ber, C3vAcear, zeXocstnvN'yNBakAl:
ngV hL;jWHO:zpMIwtheakst 3-kSoo;irt?!Iohearl

SlqfoIY,upeCufgRRJ.oTFVkbjZj artl:
eOhe 

qBoIHFIC,enimeakmeaIRMJPWrABR;-U&qG$GK xxxoc
.

gQi&i:Zg:!;ze KZemeeat vB:R:qGn:&HJRFWyceat heVl:PiOx!llDksn:OkXpr,N;jKGfoYn:
:


qnEKhp3gl
BLXbme :VxExXNqg$pe Cv'Fr CxcprExznW-kWO--WXLoUz3kst aGLZycs,Xw 3E,jLVxOmet CireYYt d hl:Vame!l:LDBNV&kAl:heatnUfor wwl:;Mr::DoI?yvqETFstVw:QBogl:hee CXPX&O
Seme fWQFWMQFfpeakQr C!heXoDdTqs?X.X'GBoXHfoMYspe?H:dhok;j


SAbJDat Cir Wspe s.t n:
bYSkst G HFAq!yher CN
Sorso;zl
A,ukuncVNJ-W